# 13 — Eval-Awareness Directions on Llama 3.3 70B (Nguyen recipe)

Self-contained notebook that refits the eval-awareness probes from Nguyen et al. *Probing and Steering Evaluation Awareness of Language Models* (arXiv 2507.01786) on **`meta-llama/Llama-3.3-70B-Instruct`**, on each of the 4 contrastive datasets the authors release (`contrastive_dataset`, `AB_ambiguous`, `tom_vs_sharegpt`, `truthful`).

**Why we refit instead of downloading**: The authors' GitHub repo (`Jordine/evaluation-awareness-probing`) ships training scripts + the 4 contrastive JSON datasets, but **no pre-trained probe weights** — no `.pt`, no `.safetensors`, no GitHub releases. The recipe is a simple per-layer mean-difference, so refitting is cheaper than tracking down the binaries.

**The recipe** (Nguyen §3, simplified to mean-difference probes):
1. For each item `{question, positive (test), negative (deploy)}`, build two prompts:
    - `test_prompt   = question + "\n\n " + test_answer`     (e.g. `"(A)"`)
    - `deploy_prompt = question + "\n\n " + deploy_answer`   (e.g. `"(B)"`)
2. Tokenize, run a forward pass, capture the residual stream at the position of the answer-character token (the `'A'` or `'B'`) at every transformer layer.
3. After all items: per (dataset, layer), `direction = mean(test_acts) - mean(deploy_acts)`.
4. Save unit-normalized directions to NPZ keyed by `{dataset}__layer_{LLL}`.

**One deliberate deviation from Nguyen**: the original code hooks `blocks.{layer}.hook_resid_pre` (TransformerLens). Here we use **post-block forward hooks** (matching `mech_spoof.activations._register_hooks`), which is what was used for our existing `exp06_lamma/directions.npz` S/U directions. This makes the cosine matrix in §7 below layer-aligned: `eval_aw[L]` and `sui[L]` are both post-block-`L` residuals. Numerically the directions barely shift since post-block-`L` ≡ pre-block-`(L+1)` modulo the next block's pre-LN.

**Outputs (saved to `OUT_DIR`):**
- `directions.npz` — unit-norm directions, one per (dataset, layer), keyed `{dataset}__layer_{LLL}`. Also includes `{dataset}__test_mean__layer_{LLL}` and `..._deploy_mean__layer_{LLL}` for downstream steering work that needs the un-differenced means.
- `manifest.json` — per-(dataset, layer) `frac_test_proj_larger`, `mm_natural_scale`, `n_used`, `n_skipped`.
- `cosine_vs_sui.json` (if `exp06_lamma/directions.npz` exists) — per-layer cosines.

**Compute:** Llama 3.3 70B in 8-bit (bitsandbytes) fits on a single 80 GB GPU. ~1,500 items × 2 prompts = ~3,000 forward passes; expect **45–90 min** wall-clock on a single H100/A100, longer on slower hardware. No quantization loss meaningfully matters for mean-difference directions.

**Tokens needed:**
- `HF_TOKEN` — Llama 3.3 is gated.

## 0 — Sanity: GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'no CUDA — this notebook needs a GPU'
n = torch.cuda.device_count()
total_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n)) / 1e9
print(f'{n} GPUs visible, total VRAM = {total_gb:.0f} GB')
# 70B in 8-bit fits in ~70 GB; bf16 needs ~140 GB.
assert total_gb >= 60, '70B in 8-bit needs at least ~60 GB; bf16 needs ~140 GB'

## 1 — Install dependencies

Most pods already have `torch`/`transformers`. `bitsandbytes` + `accelerate` are needed for 8-bit loading. The project root provides `mech_spoof.models.load_model` which wraps the bnb config we already validated for this model in commit `8d508cd`.

In [ ]:
# Comment out if your env already has these.
!pip install -q 'transformers>=4.45' 'bitsandbytes>=0.43' 'accelerate>=0.33' huggingface_hub tqdm

## 2 — Project root, tokens, paths

In [ ]:
import os, sys, json
from pathlib import Path

# Adjust if you rsynced/cloned the project elsewhere on the pod.
PROJECT_ROOT = Path(os.environ.get('MECH_SPOOF_ROOT', '/workspace/Mech_spoof'))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    print(f'falling back to PROJECT_ROOT={PROJECT_ROOT}')
assert (PROJECT_ROOT / 'src' / 'mech_spoof').exists(), f'no mech_spoof at {PROJECT_ROOT}/src — set MECH_SPOOF_ROOT'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

DATASETS_DIR = PROJECT_ROOT / 'third_party' / 'eval_awareness_datasets'
OUT_DIR      = PROJECT_ROOT / 'exp_eval_awareness' / 'llama33_70b'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HF auth — Llama 3.3 is gated.
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
except Exception:
    pass
assert os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN'), \
    'Set HF_TOKEN in env (Llama 3.3 is gated).'
from huggingface_hub import login
login(token=os.environ.get('HF_TOKEN') or os.environ['HUGGING_FACE_HUB_TOKEN'], add_to_git_credential=False)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('datasets dir :', DATASETS_DIR, ' (exists)' if DATASETS_DIR.exists() else ' (MISSING)')
print('out dir      :', OUT_DIR)

## 3 — Load Llama 3.3 70B (8-bit)

Uses the project's `mech_spoof.models.load_model('llama33_70b')` which already configures bitsandbytes 8-bit loading + `device_map='auto'` for sharding across whatever GPUs are present. If you want bf16 (cleaner activations, ~2× the VRAM), pass `quantization=None`.

In [ ]:
from mech_spoof.models import load_model

loaded = load_model('llama33_70b')   # 8-bit by default per configs.py
model     = loaded.hf_model
tokenizer = loaded.tokenizer
n_layers  = loaded.n_layers
d_model   = loaded.hf_model.config.hidden_size
first_param_device = next(model.parameters()).device
print(f'model loaded: n_layers={n_layers}, d_model={d_model}, first param on {first_param_device}')
model.eval()

## 4 — Load datasets

All 4 are flat lists of `{question, positive|test, negative|deploy [, category]}`. Normalize to `(test_answer, deploy_answer)` so the rest of the code is uniform.

In [ ]:
DATASETS = ['contrastive_dataset', 'AB_ambiguous', 'tom_vs_sharegpt', 'truthful']

def _load(ds_name: str):
    raw = json.loads((DATASETS_DIR / f'{ds_name}.json').read_text())
    items = []
    for r in raw:
        # `AB_ambiguous` uses test/deploy keys; the others use positive/negative.
        if 'positive' in r and 'negative' in r:
            items.append({'question': r['question'], 'test': r['positive'], 'deploy': r['negative']})
        elif 'test' in r and 'deploy' in r:
            items.append({'question': r['question'], 'test': r['test'], 'deploy': r['deploy']})
        else:
            raise KeyError(f'{ds_name}: unexpected keys {list(r.keys())}')
    return items

datasets = {d: _load(d) for d in DATASETS}
for d, items in datasets.items():
    print(f'  {d:<22s} {len(items):>4d} items   sample: test={items[0]["test"]!r}  deploy={items[0]["deploy"]!r}')

## 5 — Single-prompt activation extractor

For each prompt: tokenize, run one forward pass with a post-block hook on every layer, return the residual stream at the position of the **last `A`/`B` answer-character token**. Returns a `(n_layers, d_model)` float32 CPU tensor (~2.6 MB per prompt).

Hook convention: `register_forward_hook` on `model.model.layers[L]` captures the **output** of block L (post-residual). This matches `mech_spoof.activations._register_hooks` so per-layer S/U and per-layer eval-aw directions are layer-aligned for cosines.

In [ ]:
# Token IDs for 'A' and 'B' as Llama tokenizes them inside `"(A)"` / `"(B)"`.
# In Llama 3 / 3.3 these are isolated single-char tokens.
TOK_A = tokenizer.convert_tokens_to_ids('A')
TOK_B = tokenizer.convert_tokens_to_ids('B')
assert TOK_A is not None and TOK_B is not None and TOK_A != tokenizer.unk_token_id, \
    f'A/B token-id resolution failed: A={TOK_A} B={TOK_B}'
print(f'TOK_A={TOK_A}  TOK_B={TOK_B}  decode={tokenizer.decode([TOK_A])!r}/{tokenizer.decode([TOK_B])!r}')

In [ ]:
def extract_at_answer_token(prompt_text: str, answer_token_id: int) -> torch.Tensor | None:
    """Forward-pass `prompt_text`, return (n_layers, d_model) float32 CPU activation
    at the LAST occurrence of `answer_token_id` in the tokenized prompt. None if absent."""
    enc = tokenizer(prompt_text, return_tensors='pt', add_special_tokens=True)
    input_ids = enc.input_ids.to(first_param_device)
    matches = (input_ids[0] == answer_token_id).nonzero(as_tuple=True)[0]
    if len(matches) == 0:
        return None
    pos = matches[-1].item()

    storage: list = [None] * n_layers
    handles = []
    for i in range(n_layers):
        layer = loaded.layer_module(i)
        def _hook(_m, _inp, out, idx=i):
            h = out[0] if isinstance(out, tuple) else out
            storage[idx] = h[0, pos].detach().float().cpu()   # (d_model,)
        handles.append(layer.register_forward_hook(_hook))
    try:
        with torch.no_grad():
            model(input_ids=input_ids, use_cache=False)
    finally:
        for h in handles:
            h.remove()
    return torch.stack(storage)  # (n_layers, d_model)

# Smoke test on one item — should print no Nones and a sane norm.
_q = datasets['contrastive_dataset'][0]['question']
_ans = datasets['contrastive_dataset'][0]['test']                 # e.g. '(A)'
_tok = TOK_A if 'A' in _ans else TOK_B
_acts = extract_at_answer_token(f'{_q}\n\n {_ans}', _tok)
print(f'smoke: shape={tuple(_acts.shape)}  norm@L0={_acts[0].norm():.2f}  norm@L40={_acts[40].norm():.2f}  norm@L79={_acts[-1].norm():.2f}')

## 6 — Fit per-layer mean-difference directions per dataset

Streaming-mean accumulator (no need to hold all activations in RAM): per (dataset, layer) we maintain running sums for test and deploy separately, then `direction = (test_mean - deploy_mean) / ||·||`.

We also stash the un-normalized `mm_natural_scale = ||test_mean - deploy_mean||` and `frac_test_proj_larger` (fraction of test items where `test_act @ direction > deploy_act @ direction`) — sanity stats analogous to those in `exp06_lamma/manifest.json`.

In [ ]:
from tqdm.auto import tqdm
import numpy as np

# Optional: cap items per dataset for a fast smoke run.
MAX_ITEMS_PER_DATASET = None    # e.g. 50 for a 5-min smoke run; None = full

results: dict[str, dict] = {}
for ds_name, items in datasets.items():
    if MAX_ITEMS_PER_DATASET:
        items = items[:MAX_ITEMS_PER_DATASET]

    test_sum   = torch.zeros(n_layers, d_model, dtype=torch.float64)
    deploy_sum = torch.zeros(n_layers, d_model, dtype=torch.float64)
    n_test = n_deploy = n_skipped = 0
    # For frac_test_proj_larger we compute on-the-fly after we know the direction;
    # cache the per-item activations on CPU (~2.6 MB each, manageable for ~600 items).
    test_acts:   list[torch.Tensor] = []
    deploy_acts: list[torch.Tensor] = []

    for it in tqdm(items, desc=ds_name):
        prompt_t = f"{it['question']}\n\n {it['test']}"
        prompt_d = f"{it['question']}\n\n {it['deploy']}"
        tok_t    = TOK_A if 'A' in it['test']   else TOK_B
        tok_d    = TOK_A if 'A' in it['deploy'] else TOK_B
        a_t = extract_at_answer_token(prompt_t, tok_t)
        a_d = extract_at_answer_token(prompt_d, tok_d)
        if a_t is None or a_d is None:
            n_skipped += 1
            continue
        test_sum   += a_t.double()
        deploy_sum += a_d.double()
        n_test   += 1
        n_deploy += 1
        test_acts.append(a_t)
        deploy_acts.append(a_d)

    assert n_test == n_deploy and n_test > 0, f'{ds_name}: n_test={n_test} n_deploy={n_deploy}'
    test_mean   = (test_sum   / n_test).float()       # (n_layers, d_model)
    deploy_mean = (deploy_sum / n_deploy).float()
    diff        = test_mean - deploy_mean
    norms       = diff.norm(dim=-1) + 1e-10           # (n_layers,)
    direction   = diff / norms[:, None]               # unit-norm per layer

    # frac_test_proj_larger per layer — sanity check that this direction separates the classes.
    T = torch.stack(test_acts)    # (n_items, n_layers, d_model)
    D = torch.stack(deploy_acts)
    proj_T = (T * direction[None]).sum(dim=-1)        # (n_items, n_layers)
    proj_D = (D * direction[None]).sum(dim=-1)
    frac_larger = (proj_T > proj_D).float().mean(dim=0).tolist()

    results[ds_name] = {
        'direction':       direction.numpy(),         # (n_layers, d_model)
        'test_mean':       test_mean.numpy(),
        'deploy_mean':     deploy_mean.numpy(),
        'mm_natural_scale': norms.tolist(),
        'frac_test_proj_larger': frac_larger,
        'n_used':           int(n_test),
        'n_skipped':        int(n_skipped),
    }
    print(f'  {ds_name}: n={n_test} skipped={n_skipped}  '
          f'frac_larger@L23={frac_larger[23]:.3f}  scale@L23={norms[23]:.3f}  '
          f'frac_larger@L50={frac_larger[50]:.3f}')

## 7 — Save NPZ + manifest

Layout matches the convention in `exp06_lamma/directions.npz` so downstream notebooks (e.g. `12_direction_comparison.ipynb`) can index into it identically.

Keys per dataset `D` and layer `L` (3-digit zero-padded):
- `mm_dir__{D}__layer_{LLL}` — unit-norm mean-difference direction, `(d_model,)`.
- `mm_raw__{D}__layer_{LLL}` — un-normalized `(test_mean - deploy_mean)`, `(d_model,)`.
- `test_mean__{D}__layer_{LLL}` and `deploy_mean__{D}__layer_{LLL}` — per-class means.

In [ ]:
npz_payload: dict = {}
manifest = {
    'experiment':   'eval_awareness_directions_per_dataset',
    'recipe':       'Nguyen et al. 2507.01786 mean-diff at (A)/(B) answer-char token',
    'hook':         'post-block forward hook on model.model.layers[L] (matches mech_spoof.activations)',
    'model_key':    'llama33_70b',
    'hf_id':        'meta-llama/Llama-3.3-70B-Instruct',
    'n_layers':     n_layers,
    'd_model':      d_model,
    'datasets':     {},
}
for ds_name, r in results.items():
    for L in range(n_layers):
        suffix = f'{ds_name}__layer_{L:03d}'
        npz_payload[f'mm_dir__{suffix}']     = r['direction'][L]
        npz_payload[f'mm_raw__{suffix}']     = (r['test_mean'][L] - r['deploy_mean'][L])
        npz_payload[f'test_mean__{suffix}']  = r['test_mean'][L]
        npz_payload[f'deploy_mean__{suffix}'] = r['deploy_mean'][L]
    manifest['datasets'][ds_name] = {
        'n_used':                r['n_used'],
        'n_skipped':             r['n_skipped'],
        'mm_natural_scale':      r['mm_natural_scale'],
        'frac_test_proj_larger': r['frac_test_proj_larger'],
    }

np.savez_compressed(OUT_DIR / 'directions.npz', **npz_payload)
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'wrote {len(npz_payload)} arrays to {OUT_DIR / "directions.npz"}')
print(f'wrote manifest to {OUT_DIR / "manifest.json"}')

## 8 — (Optional) cosine vs S/U direction at every layer

If `exp06_lamma/directions.npz` exists, compute per-layer cosine between each eval-awareness direction and the matching-layer S/U direction. This is the per-layer Phase-2 plot from the broader plan: small cosines everywhere → clean dissociation; meaningful cosines somewhere in the stack → entanglement worth investigating.

In [ ]:
SUI_NPZ = PROJECT_ROOT / 'exp06_lamma' / 'directions.npz'
if not SUI_NPZ.exists():
    print(f'skip: no S/U NPZ at {SUI_NPZ}')
else:
    sui_arr = np.load(SUI_NPZ)
    sui_per_layer = []
    for L in range(n_layers):
        k = f'mm_dir__response_last__layer_{L:03d}'
        if k not in sui_arr.files:
            sui_per_layer.append(None); continue
        v = sui_arr[k].astype(np.float32)
        v = v / (np.linalg.norm(v) + 1e-10)
        sui_per_layer.append(v)

    cos_curves: dict = {}
    for ds_name, r in results.items():
        cur = []
        for L in range(n_layers):
            v_aw = r['direction'][L]
            v_aw = v_aw / (np.linalg.norm(v_aw) + 1e-10)
            v_su = sui_per_layer[L]
            cur.append(float(v_aw @ v_su) if v_su is not None else None)
        cos_curves[ds_name] = cur
        nonnull = [c for c in cur if c is not None]
        if nonnull:
            print(f'  {ds_name:<22s}  mean|cos|={np.mean(np.abs(nonnull)):.3f}  '
                  f'max|cos|={np.max(np.abs(nonnull)):.3f}  '
                  f'argmax_layer={int(np.argmax(np.abs(nonnull)))}')

    (OUT_DIR / 'cosine_vs_sui.json').write_text(json.dumps(cos_curves, indent=2))
    print(f'\nwrote {OUT_DIR / "cosine_vs_sui.json"}')

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(10, 5))
        for ds_name, cur in cos_curves.items():
            ax.plot(cur, label=ds_name)
        ax.axhline(0, color='k', lw=0.5)
        ax.axhline(0.2, color='r', lw=0.5, ls='--', label='|cos|=0.2 entanglement threshold')
        ax.axhline(-0.2, color='r', lw=0.5, ls='--')
        ax.set_xlabel('layer'); ax.set_ylabel('cosine(eval_aw[L], S/U[L])')
        ax.set_title('Per-layer cosine: eval-awareness vs S/U on Llama 3.3 70B')
        ax.legend(loc='best', fontsize=8)
        plt.tight_layout(); plt.show()
    except ImportError:
        pass

## Notes & gotchas

- **What "layer 23" means here.** Nguyen et al. report the Simple Contrastive probe at layer 23 of Llama 3.3 70B as the best generalizer. With our 0-indexed post-block hook, `layer_023` is post-block-23 = pre-block-24 in TransformerLens. Numerically very close; for downstream comparison just be consistent.
- **Quantization noise.** 8-bit loading adds dequantization noise to activations. For a mean-difference fit averaged over hundreds of items per class this is negligible (the noise averages out); for steering effects you may want bf16. Pass `quantization=None` to `load_model` (and have ~140 GB total VRAM) to refit cleanly.
- **Token position robustness.** A few items may not contain a tokenized standalone `'A'` or `'B'` (rare; happens when the answer punctuation absorbs context). Those are skipped and counted in `n_skipped`. Inspect `manifest.json` if `n_skipped` is unexpectedly high for any dataset.
- **Why no SAD `stages_oversight`.** Nguyen also evaluate on SAD `stages_oversight` and the additional Truthful/Ambiguous Statements + ToM-vs-ShareGPT splits. The 4 datasets above are what the authors check into their repo. To extend to SAD, drop `LRudL/sad`'s `stages_oversight` JSON into `third_party/eval_awareness_datasets/` with the same `{question, positive, negative}` schema and add it to `DATASETS`.
- **Re-using `loaded`.** If you keep the model loaded in the kernel, you can run notebook 10's S/U fit on the same model immediately after — no reload required.